## ライブラリの読み込み

In [ ]:
import random
from beamngpy import BeamNGpy, Scenario, Vehicle, set_up_simple_logging
from beamngpy.sensors import Timer
from pathlib import Path
import json
from datetime import datetime
import time
import matplotlib.pyplot as plt
import numpy as np
import threading

## BeamNGの起動

In [ ]:
random.seed(1703)
set_up_simple_logging()

beamng = BeamNGpy('localhost', 64256)
bng = beamng.open(launch=False)

In [ ]:
vehicle_model = 'etk800'
map_name = 'c1'
spawn_pos = (3819.65, -5113.19, 852.5)
spawn_rot = (0.0, 0.0, -0.25881905, 0.96592583)

In [ ]:
vehicle = Vehicle('ego_vehicle', model=vehicle_model, licence='ego_vehicle')
timer = Timer()

vehicle.sensors.attach('timer', timer)

scenario = Scenario(map_name, 'LiDAR_demo', description='Spanning the map with a LiDAR sensor')

# Add the vehicle to the scenario with the specified initial position and orientation  
scenario.add_vehicle(vehicle, cling=True,
  pos=spawn_pos,  # Initial position (x, y, z)  
  rot_quat=spawn_rot  # Initial orientation as a quaternion (x, y, z, w)  
)

scenario.make(bng)
bng.settings.set_deterministic(60)
bng.load_scenario(scenario)
bng.ui.hide_hud()
bng.scenario.start()

In [ ]:
vehicle.sensors.poll()
print(vehicle.sensors['state'])
print(vehicle.sensors['timer'])

## Trajectoryの記録

In [ ]:
class TrajectoryRecorder:
  def __init__(self, vehicle, metadata=None):
    self.vehicle = vehicle
    self.trajectory = []
    self.metadata = metadata or {}
    self._initialized = False  # 初回センサ値を無視する

  def record_frame(self):
    """1フレーム分のデータを記録"""
    self.vehicle.sensors.poll()
    state_data = self.vehicle.sensors['state']
    timer_data = self.vehicle.sensors['timer']

    # 初回 poll のセンサ値は古い可能性があるので無視する
    if not self._initialized:
      self._initialized = True
      return None  # このフレームは捨てる

    frame = {
      'x': state_data['pos'][0],
      'y': state_data['pos'][1],
      'z': state_data['pos'][2],
      't': timer_data['time']
    }

    self.trajectory.append(frame)
    return frame
  
  def save(self, filename, output_dir='data/scripts', tol=0.01):
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    filepath = output_path / filename

    # prevとframeを比較してtol以上動いた点だけを抽出
    corrected = []

    # 最初の移動点を探す
    move_idx = None
    for i in range(len(self.trajectory) - 1):
        p1 = self.trajectory[i]
        p2 = self.trajectory[i + 1]
        dist = ((p2['x'] - p1['x'])**2 +
                (p2['y'] - p1['y'])**2 +
                (p2['z'] - p1['z'])**2)**0.5
        if dist >= tol:
            move_idx = i
            break
        
    t0 = self.trajectory[move_idx]['t']
    prev = self.trajectory[move_idx]
    prev['t'] -= t0
    corrected.append(prev)

    # 開始地点と終了地点での停止点を消すための処理
    for i in self.trajectory[move_idx:]:
      dist = ((i['x'] - prev['x'])**2 +
              (i['y'] - prev['y'])**2 +
              (i['z'] - prev['z'])**2)**0.5
      if dist >= tol:
        i['t'] -= t0
        corrected.append(i)
        prev = i

    output_data = {
      'metadata': {
        'created_at': datetime.now().isoformat(),
        'total_waypoints': len(corrected),
        'duration': corrected[-1]['t'] if corrected else 0,
        **self.metadata
      },
      'script': corrected
    }

    with open(filepath, 'w', encoding='utf-8') as f:
      json.dump(output_data, f, indent=2)

    print(f"✓ 保存完了: {filepath}, ウェイポイント数: {len(corrected)}")
    return str(filepath)
  
  def get_summary(self):
    """記録データのサマリーを取得"""
    if not self.trajectory:
      return None
    
    distance = 0
    for i in range(len(self.trajectory) - 1):
      p1 = self.trajectory[i]
      p2 = self.trajectory[i + 1]
      distance += ((p2['x'] - p1['x'])**2 +
                   (p2['y'] - p1['y'])**2 +
                   (p2['z'] - p1['z'])**2)**0.5
    
    return {
      'total_waypoints': len(self.trajectory),
      'duration': self.trajectory[-1]['t'],
      'distance': distance,
      'avg_speed': distance / self.trajectory[-1]['t'] if self.trajectory[-1]['t'] > 0 else 0,
      'start_position': (
        self.trajectory[0]['x'],
        self.trajectory[0]['y'],
        self.trajectory[0]['z']
      ),
      'end_position': (
        self.trajectory[-1]['x'],
        self.trajectory[-1]['y'],
        self.trajectory[-1]['z']
      )
    }

In [ ]:
recorder = TrajectoryRecorder(vehicle, metadata={
  'map': map_name,
  'vehicle_model': vehicle_model,
  'spawn_pos': spawn_pos,
  'spawn_rot': spawn_rot
})

In [ ]:
stop_flag = False

def wait_for_enter():
  global stop_flag
  input()   # Enter を待つだけ
  stop_flag = True

print("Enter を押すと記録開始します...")
input()  # ← 開始

print("記録開始！ Enter を押すと終了します")

recorder.trajectory = []  # 必要に応じてリセット

# Enter を待つスレッドを起動
listener = threading.Thread(target=wait_for_enter, daemon=True)
listener.start()

while True:
  if stop_flag:
    print("記録終了")
    break

  frame = recorder.record_frame()

  time.sleep(0.1)

In [ ]:
summary = recorder.get_summary()
print("\n記録サマリー:")
print(f"  移動距離: {summary['distance']:.1f}m")
print(f"  平均速度: {summary['avg_speed']:.1f}m/s ({summary['avg_speed']*3.6:.1f}km/h)")

# 保存（これだけでOK）
recorder.save('highway_trajectory_001.json')

## Trajectoryの可視化

In [ ]:
def quat_conjugate(q):
  return np.array([-q[0], -q[1], -q[2], q[3]])

def quat_mul(q1, q2):
  x1, y1, z1, w1 = q1
  x2, y2, z2, w2 = q2
  return np.array([
    w1*x2 + x1*w2 + y1*z2 - z1*y2,
    w1*y2 - x1*z2 + y1*w2 + z1*x2,
    w1*z2 + x1*y2 - y1*x2 + z1*w2,
    w1*w2 - x1*x2 - y1*y2 - z1*z2
  ])

def rotate_vector(v, q):
  # v を quaternion q で回転
  v_quat = np.array([v[0], v[1], v[2], 0.0])
  return quat_mul(quat_mul(q, v_quat), quat_conjugate(q))[:3]

In [ ]:
# ---- Load JSON file ----
json_file = "/home/apollo-22/offroad/beamng_data_collector/route_planning/data/scripts/highway_trajectory_001.json"
with open(json_file, "r") as f:
  data = json.load(f)

spawn_pos = np.array(data["metadata"]["spawn_pos"])
spawn_rot = np.array(data["metadata"]["spawn_rot"])  # [x,y,z,w] quaternion

# ---- Transform all waypoints ----
xs = []
ys = []

for pt in data["script"]:
  pos = np.array([pt["x"], pt["y"], pt["z"]])

  # 平行移動（車両位置を原点へ）
  relative = pos - spawn_pos

  # 回転（車両の向きを +X に合わせる）
  rotated = rotate_vector(relative, quat_conjugate(spawn_rot))

  xs.append(rotated[0])
  ys.append(rotated[1])

# ---- Plot ----
plt.figure(figsize=(8, 8))
plt.plot(xs, ys, marker=".", linewidth=1)
plt.title("Route (Vehicle Local Frame)")
plt.xlabel("X (forward)")
plt.ylabel("Y (left)")
plt.axis("equal")
plt.grid(True)
plt.show()
